# 🏢 CompanyOS — GRPO Training Notebook

**Environment:** CompanyOS on HuggingFace Spaces  
**Method:** GRPO via HuggingFace TRL + Unsloth  
**Works on:** HuggingFace Spaces GPU (ZeroGPU / credits) + Google Colab  

---
### How HF Credits Work
- If running on **HF Spaces with GPU** (A10G / T4 via ZeroGPU or paid credits): run this notebook directly in the Space
- If running on **Colab**: set `ENV_URL` to your deployed HF Space URL
- The env always runs on HF Spaces — only the **training compute** changes

---

## 0. Detect Runtime (HF Spaces vs Colab)

In [ ]:
import os
import sys

# ── Detect where we're running ─────────────────────────────────────────────
IS_HF_SPACE  = os.path.exists('/home/user')          # HuggingFace Spaces
IS_COLAB     = 'google.colab' in sys.modules         # Google Colab
IS_LOCAL     = not IS_HF_SPACE and not IS_COLAB      # Local machine

print(f'Runtime: {"HF Spaces" if IS_HF_SPACE else "Colab" if IS_COLAB else "Local"}')

# ── Config — edit these ────────────────────────────────────────────────────
ENV_URL       = 'https://satyamshahi-companyos.hf.space'  # your HF Space URL
MODEL_NAME    = 'unsloth/Qwen2.5-1.5B-Instruct'           # small fast model
HF_TOKEN      = os.environ.get('HF_TOKEN', '')            # set in HF Space secrets
WANDB_TOKEN   = os.environ.get('WANDB_API_KEY', '')       # optional

# ── Training hyperparams ───────────────────────────────────────────────────
MAX_EPISODES        = 200    # increase to 500 for better convergence
GRPO_UPDATE_EVERY   = 10     # run GRPO update every N episodes
GRPO_BATCH_SIZE     = 4
GRPO_GRAD_ACCUM     = 4
LEARNING_RATE       = 5e-6
MAX_NEW_TOKENS      = 150
LORA_R              = 16

print(f'ENV_URL: {ENV_URL}')
print(f'Model:   {MODEL_NAME}')
print(f'Episodes: {MAX_EPISODES}')

## 1. Install Dependencies

In [ ]:
# ── Install based on runtime ───────────────────────────────────────────────
if IS_HF_SPACE:
    # HF Spaces: use uv for fast installs (pre-installed on most spaces)
    os.system('pip install -q unsloth trl transformers datasets requests matplotlib wandb')
else:
    # Colab / Local
    os.system('pip install -q unsloth trl transformers datasets requests matplotlib wandb')

print('Dependencies installed.')

## 2. Login to HuggingFace (needed for model download + pushing trained model)

In [ ]:
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Logged in to HuggingFace via token.')
else:
    # Interactive login — works on Colab
    print('No HF_TOKEN found. If on HF Spaces, set it in Settings > Secrets.')
    print('If on Colab, run: from huggingface_hub import login; login()')

## 3. Verify Environment is Reachable

In [ ]:
import requests
import json

# ── Health check ───────────────────────────────────────────────────────────
health = requests.get(f'{ENV_URL}/health', timeout=30).json()
print('Health:', health)

# ── Reset test ─────────────────────────────────────────────────────────────
obs = requests.post(f'{ENV_URL}/reset', json={}, timeout=30).json()
print('Task:', obs['observation']['task'])
print('Tools:', list(obs['observation']['tools'].keys()))
print('Progress keys:', list(obs['observation']['progress'].keys()))
print('\n✅ Environment reachable and working!')

## 4. Environment Client

In [ ]:
# ── Simple REST client for the CompanyOS env ──────────────────────────────

def env_reset(task_id=None, seed=None):
    body = {}
    if task_id: body['task_id'] = task_id
    if seed:    body['seed']    = seed
    r = requests.post(f'{ENV_URL}/reset', json=body, timeout=30)
    return r.json()['observation']

def env_step(app, method, params=None):
    body = {'app': app, 'method': method, 'params': params or {}}
    r = requests.post(f'{ENV_URL}/step', json=body, timeout=30)
    d = r.json()
    return d['observation'], d['reward'], d['done'], d['info']

print('Environment client ready.')

## 5. Random Baseline — BEFORE Training

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

RANDOM_ACTIONS = [
    ('ticketdesk',    'list_tickets',    {}),
    ('ticketdesk',    'search_tickets',  {'query': 'vendor'}),
    ('ticketdesk',    'get_ticket',      {'ticket_id': 'T-001'}),
    ('ticketdesk',    'update_ticket',   {'ticket_id': 'T-001', 'field': 'priority',  'value': 'high'}),
    ('ticketdesk',    'update_ticket',   {'ticket_id': 'T-001', 'field': 'verified',  'value': True}),
    ('ticketdesk',    'update_ticket',   {'ticket_id': 'T-003', 'field': 'status',    'value': 'open'}),
    ('datahub',       'list_metrics',    {}),
    ('datahub',       'query_metric',    {'metric_name': 'vendor_compliance_score'}),
    ('datahub',       'refresh_data',    {'metric_name': 'vendor_compliance_score'}),
    ('datahub',       'get_approver',    {'role': 'CFO'}),
    ('datahub',       'get_approver',    {'role': 'CFO_DELEGATE'}),
    ('approvalflow',  'list_approval_types', {}),
    ('approvalflow',  'submit_approval', {'approval_type': 'vendor_onboarding', 'approver': 'david.kim',
                                          'data': {'vendor_name': 'ACME', 'compliance_score': 85}}),
    ('approvalflow',  'check_status',    {'approval_id': 'APR-001'}),
    # Bad actions — random agent doesn't know better
    ('ticketdesk',    'update_ticket',   {'ticket_id': 'T-003', 'field': 'status', 'value': 'closed'}),
    ('approvalflow',  'submit_approval', {'approval_type': 'vendor_onboarding', 'approver': 'sarah.chen',
                                          'data': {'vendor_name': 'ACME', 'compliance_score': 85}}),
]

BASELINE_EPISODES = 100
baseline_rewards   = []
baseline_successes = []

print(f'Running random baseline for {BASELINE_EPISODES} episodes...')
for ep in range(BASELINE_EPISODES):
    obs      = env_reset()
    done     = False
    ep_reward = 0.0
    while not done:
        app, method, params = random.choice(RANDOM_ACTIONS)
        _, reward, done, info = env_step(app, method, params)
        ep_reward += reward
    baseline_rewards.append(ep_reward)
    baseline_successes.append(info.get('success', False))
    if (ep + 1) % 20 == 0:
        mean_r = sum(baseline_rewards[-20:]) / 20
        sr     = sum(baseline_successes[-20:]) / 20
        print(f'  Ep {ep+1:>4} | avg_reward={mean_r:+.2f} | success_rate={sr:.0%}')

print(f'\n── Baseline Results ─────────────────────')
print(f'  Mean reward:  {sum(baseline_rewards)/len(baseline_rewards):+.2f}')
print(f'  Success rate: {sum(baseline_successes)/len(baseline_successes):.1%}')
print(f'  Best episode: {max(baseline_rewards):+.2f}')

## 6. Load Model with Unsloth

In [ ]:
import torch
from unsloth import FastLanguageModel

print(f'Loading {MODEL_NAME}...')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow!)"}') 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit   = True,   # saves VRAM — works on T4 and A10G
    token          = HF_TOKEN if HF_TOKEN else None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    target_modules = ['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_alpha     = LORA_R,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',  # saves memory
)

print(f'Model loaded. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 7. Agent: Observation → Action via LLM

In [ ]:
import re

SYSTEM_PROMPT = """You are an enterprise workflow agent navigating CompanyOS.
Complete the given task by calling tools across three apps:

ticketdesk:   list_tickets | search_tickets(query) | get_ticket(ticket_id) | update_ticket(ticket_id, field, value)
datahub:      list_metrics | query_metric(metric_name) | refresh_data(metric_name) | get_approver(role)
approvalflow: list_approval_types | submit_approval(approval_type, approver, data) | check_status(approval_id) | escalate(approval_id, reason)

Key rules:
- Always refresh stale data before using it
- Check get_approver() before submitting approvals — the CFO may be OOO
- Verify a ticket before linking an approval
- Poll check_status() until the approval resolves

Respond ONLY with a single JSON object. No explanation. No markdown. Example:
{"app": "ticketdesk", "method": "get_ticket", "params": {"ticket_id": "T-001"}}"""

def obs_to_prompt(obs):
    return (
        f"Task: {obs['task']}\n"
        f"Step: {obs['step']}/{obs['max_steps']} (remaining: {obs['steps_remaining']})\n"
        f"Progress: {json.dumps(obs['progress'])}\n"
        f"Last result: {json.dumps(obs.get('last_result', {}))}\n"
        f"\nWhat is your next action?"
    )

def parse_action(text):
    """Parse JSON action from model output. Falls back to safe default."""
    try:
        match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
        if match:
            action = json.loads(match.group())
            if action.get('app') and action.get('method'):
                return action
    except Exception:
        pass
    return {'app': 'ticketdesk', 'method': 'list_tickets', 'params': {}}

def agent_act(obs, temperature=0.7):
    """Sample an action from the model given the current observation."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': obs_to_prompt(obs)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            inputs,
            max_new_tokens  = MAX_NEW_TOKENS,
            temperature     = temperature,
            do_sample       = True,
            pad_token_id    = tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    return parse_action(text), text

print('Agent ready.')

## 8. GRPO Training Loop

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import time

# ── Training state ─────────────────────────────────────────────────────────
train_rewards    = []   # per-episode total reward
train_successes  = []   # per-episode success flag
train_steps      = []   # steps taken per episode
grpo_losses      = []   # loss after each GRPO update

all_prompts      = []
all_completions  = []
all_step_rewards = []

print(f'Starting GRPO training for {MAX_EPISODES} episodes...')
print(f'GRPO update every {GRPO_UPDATE_EVERY} episodes')
print('─' * 60)

start_time = time.time()

for episode in range(MAX_EPISODES):
    obs       = env_reset()
    done      = False
    ep_reward = 0.0
    ep_prompts, ep_completions, ep_rewards = [], [], []

    while not done:
        prompt     = obs_to_prompt(obs)
        action, completion = agent_act(obs)

        obs, reward, done, info = env_step(
            action.get('app',    'ticketdesk'),
            action.get('method', 'list_tickets'),
            action.get('params', {})
        )

        ep_prompts.append(prompt)
        ep_completions.append(json.dumps(action))
        ep_rewards.append(reward)
        ep_reward += reward

    # Record episode stats
    train_rewards.append(ep_reward)
    train_successes.append(info.get('success', False))
    train_steps.append(info.get('step', 0))

    # Accumulate for GRPO batch
    all_prompts.extend(ep_prompts)
    all_completions.extend(ep_completions)
    all_step_rewards.extend(ep_rewards)

    # ── GRPO update every N episodes ──────────────────────────────────────
    if (episode + 1) % GRPO_UPDATE_EVERY == 0:
        # Use last 200 transitions for the update
        batch_size = min(200, len(all_prompts))
        ds = Dataset.from_dict({
            'prompt':     all_prompts[-batch_size:],
            'completion': all_completions[-batch_size:],
            'reward':     all_step_rewards[-batch_size:],
        })

        grpo_config = GRPOConfig(
            output_dir                  = '/tmp/companyos_grpo',
            num_train_epochs            = 1,
            per_device_train_batch_size = GRPO_BATCH_SIZE,
            gradient_accumulation_steps = GRPO_GRAD_ACCUM,
            learning_rate               = LEARNING_RATE,
            logging_steps               = 1,
            save_strategy               = 'no',
            report_to                   = 'none',
            fp16                        = torch.cuda.is_available(),
        )
        trainer = GRPOTrainer(
            model        = model,
            tokenizer    = tokenizer,
            config       = grpo_config,
            train_dataset= ds,
        )
        train_result = trainer.train()
        grpo_losses.append(train_result.training_loss)

        # Print progress
        mean_r  = sum(train_rewards[-GRPO_UPDATE_EVERY:]) / GRPO_UPDATE_EVERY
        sr      = sum(train_successes[-GRPO_UPDATE_EVERY:]) / GRPO_UPDATE_EVERY
        elapsed = (time.time() - start_time) / 60
        print(
            f'Ep {episode+1:>4} | '
            f'avg_reward={mean_r:+6.2f} | '
            f'success={sr:.0%} | '
            f'loss={grpo_losses[-1]:.4f} | '
            f'{elapsed:.1f}min'
        )

print('\nTraining complete!')

## 9. Learning Curves 📈

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def smooth(data, window=10):
    if len(data) < window:
        return data
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig = plt.figure(figsize=(16, 10))
fig.suptitle('CompanyOS — Training Results', fontsize=15, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

WINDOW = 10
ep_range_smooth = range(WINDOW - 1, len(train_rewards))

# ── Plot 1: Reward curve — baseline vs trained ──────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
# Baseline (horizontal band)
baseline_mean = sum(baseline_rewards) / len(baseline_rewards)
baseline_std  = np.std(baseline_rewards)
ax1.axhspan(baseline_mean - baseline_std, baseline_mean + baseline_std,
            alpha=0.15, color='tomato', label='Random baseline range')
ax1.axhline(baseline_mean, color='tomato', linestyle='--', linewidth=1.5,
            label=f'Random baseline mean ({baseline_mean:.1f})')
# Training rewards
ax1.plot(train_rewards, alpha=0.2, color='steelblue')
ax1.plot(list(ep_range_smooth), smooth(train_rewards, WINDOW),
         color='steelblue', linewidth=2.5, label=f'Trained agent (smoothed w={WINDOW})')
ax1.set_title('Episode Reward: Random Baseline vs Trained Agent', fontsize=12)
ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Reward')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# ── Plot 2: Success rate ────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
success_vals = [1.0 if s else 0.0 for s in train_successes]
baseline_sr  = sum(1 for s in baseline_successes if s) / len(baseline_successes)
ax2.axhline(baseline_sr, color='tomato', linestyle='--', linewidth=1.5,
            label=f'Baseline SR ({baseline_sr:.1%})')
if len(success_vals) >= WINDOW:
    ax2.plot(list(ep_range_smooth), smooth(success_vals, WINDOW),
             color='seagreen', linewidth=2)
ax2.set_title('Task Success Rate')
ax2.set_xlabel('Episode')
ax2.set_ylabel('Success Rate')
ax2.set_ylim(-0.05, 1.05)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# ── Plot 3: Steps per episode ───────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(train_steps, alpha=0.2, color='mediumpurple')
if len(train_steps) >= WINDOW:
    ax3.plot(list(ep_range_smooth), smooth(train_steps, WINDOW),
             color='mediumpurple', linewidth=2)
ax3.set_title('Steps per Episode\n(lower = more efficient)')
ax3.set_xlabel('Episode')
ax3.set_ylabel('Steps')
ax3.grid(True, alpha=0.3)

# ── Plot 4: GRPO loss ───────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
if grpo_losses:
    ax4.plot(grpo_losses, color='darkorange', linewidth=2, marker='o', markersize=4)
    ax4.set_title('GRPO Training Loss')
    ax4.set_xlabel(f'Update (every {GRPO_UPDATE_EVERY} episodes)')
    ax4.set_ylabel('Loss')
    ax4.grid(True, alpha=0.3)
else:
    ax4.text(0.5, 0.5, 'No GRPO updates yet', ha='center', va='center')

plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → learning_curves.png')

## 10. Before vs After Summary Table

In [ ]:
# ── Summary stats ──────────────────────────────────────────────────────────
last_n = min(50, len(train_rewards))
trained_mean = sum(train_rewards[-last_n:]) / last_n
trained_sr   = sum(train_successes[-last_n:]) / last_n

print('=' * 50)
print('       BEFORE vs AFTER TRAINING')
print('=' * 50)
print(f'  {"Metric":<25} {"Random":>8}  {"Trained":>8}')
print('-' * 50)
print(f'  {"Mean reward":<25} {baseline_mean:>+8.2f}  {trained_mean:>+8.2f}')
print(f'  {"Success rate":<25} {baseline_sr:>8.1%}  {trained_sr:>8.1%}')
print(f'  {"Improvement (reward)":<25} {"":>8}  {trained_mean - baseline_mean:>+8.2f}')
print(f'  {"Improvement (SR)":<25} {"":>8}  {trained_sr - baseline_sr:>+8.1%}')
print('=' * 50)

## 11. Save & Push Trained Model to HF Hub

In [ ]:
# ── Save locally ───────────────────────────────────────────────────────────
model.save_pretrained('companyos-agent')
tokenizer.save_pretrained('companyos-agent')
print('Model saved locally to ./companyos-agent')

# ── Push to HF Hub (needs HF_TOKEN with write access) ─────────────────────
HF_USERNAME = 'satyamshahi'   # ← your HF username

if HF_TOKEN:
    model.push_to_hub(f'{HF_USERNAME}/companyos-agent', token=HF_TOKEN)
    tokenizer.push_to_hub(f'{HF_USERNAME}/companyos-agent', token=HF_TOKEN)
    print(f'Pushed to https://huggingface.co/{HF_USERNAME}/companyos-agent')
else:
    print('Set HF_TOKEN to push to hub.')

## 12. Quick Eval — Watch Trained Agent Play an Episode

In [ ]:
# ── Watch one episode with the trained agent ───────────────────────────────
print('=== TRAINED AGENT EPISODE ===')
obs   = env_reset(task_id='task_vendor_onboarding')
done  = False
total = 0.0

print(f'Task: {obs["task"]}\n')

while not done:
    action, raw = agent_act(obs, temperature=0.3)  # lower temp for eval
    obs, reward, done, info = env_step(
        action.get('app', ''),
        action.get('method', ''),
        action.get('params', {})
    )
    total += reward
    status = '✅' if reward > 0 else '❌'
    print(f'  Step {info["step"]:>2} | {action.get("app","")}.{action.get("method","")} → r={reward:+.2f} {status}')
    print(f'          Progress: {info["progress"]}')

outcome = '🎉 SUCCESS' if info.get('success') else '⏱️ TIMEOUT'
print(f'\n{outcome} | Total reward: {total:.2f}')